# Adversarial Robustness of BERT: Results Analysis

This notebook compares baseline vs. augmented BERT models on:
- Clean accuracy
- TextFooler attack success rate (ASR)
- Accuracy under attack

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

## 1. Load Results

In [ ]:
# Load evaluation results
with open("../results/baseline_model_eval_results.json") as f:
    baseline_eval = json.load(f)

with open("../results/augmented_model_eval_results.json") as f:
    augmented_eval = json.load(f)

# Load attack results
with open("../results/baseline_model_attack_results.json") as f:
    baseline_attack = json.load(f)

with open("../results/augmented_model_attack_results.json") as f:
    augmented_attack = json.load(f)

print("Results loaded successfully.")

## 2. Summary Table

In [ ]:
baseline_clean_acc = baseline_eval["clean_accuracy"]
augmented_clean_acc = augmented_eval["clean_accuracy"]

baseline_asr = baseline_attack["summary"]["attack_success_rate"]
augmented_asr = augmented_attack["summary"]["attack_success_rate"]

baseline_acc_under_attack = baseline_clean_acc * (1 - baseline_asr)
augmented_acc_under_attack = augmented_clean_acc * (1 - augmented_asr)

print(f"{'Metric':<30} {'Baseline':>10} {'Augmented':>10} {'Delta':>10}")
print("-" * 62)
print(f"{'Clean Accuracy':<30} {baseline_clean_acc:>10.4f} {augmented_clean_acc:>10.4f} {augmented_clean_acc - baseline_clean_acc:>+10.4f}")
print(f"{'Attack Success Rate (ASR)':<30} {baseline_asr:>10.4f} {augmented_asr:>10.4f} {augmented_asr - baseline_asr:>+10.4f}")
print(f"{'Accuracy Under Attack':<30} {baseline_acc_under_attack:>10.4f} {augmented_acc_under_attack:>10.4f} {augmented_acc_under_attack - baseline_acc_under_attack:>+10.4f}")

## 3. Comparison Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

models = ["Baseline", "Augmented"]
colors = ["#4C72B0", "#55A868"]
x = np.arange(len(models))

# Clean Accuracy
vals = [baseline_clean_acc, augmented_clean_acc]
axes[0].bar(x, vals, color=colors, width=0.5)
axes[0].set_title("Clean Accuracy", fontsize=14)
axes[0].set_xticks(x)
axes[0].set_xticklabels(models)
axes[0].set_ylim(0.8, 1.0)
for i, v in enumerate(vals):
    axes[0].text(i, v + 0.005, f"{v:.4f}", ha="center", fontsize=12)

# ASR
vals = [baseline_asr, augmented_asr]
axes[1].bar(x, vals, color=colors, width=0.5)
axes[1].set_title("Attack Success Rate (lower is better)", fontsize=14)
axes[1].set_xticks(x)
axes[1].set_xticklabels(models)
axes[1].set_ylim(0, 0.8)
for i, v in enumerate(vals):
    axes[1].text(i, v + 0.01, f"{v:.4f}", ha="center", fontsize=12)

# Accuracy Under Attack
vals = [baseline_acc_under_attack, augmented_acc_under_attack]
axes[2].bar(x, vals, color=colors, width=0.5)
axes[2].set_title("Accuracy Under Attack", fontsize=14)
axes[2].set_xticks(x)
axes[2].set_xticklabels(models)
axes[2].set_ylim(0, 0.9)
for i, v in enumerate(vals):
    axes[2].text(i, v + 0.01, f"{v:.4f}", ha="center", fontsize=12)

plt.tight_layout()
plt.savefig("../results/comparison_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to results/comparison_chart.png")

## 4. Attack Detail Analysis

In [ ]:
# Show some successful attack examples from baseline
print("=== Successful Attacks on Baseline Model (first 5) ===")
count = 0
for detail in baseline_attack["details"]:
    if detail["result_type"] == "SuccessfulAttackResult":
        print(f"\nOriginal:  {detail['original']}")
        print(f"Perturbed: {detail['perturbed']}")
        print(f"Label flip: {detail['original_output']} -> {detail['perturbed_output']}")
        count += 1
        if count >= 5:
            break

In [ ]:
# Compare attack result type distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, (name, attack_data) in enumerate([
    ("Baseline", baseline_attack),
    ("Augmented", augmented_attack),
]):
    summary = attack_data["summary"]
    sizes = [
        summary["num_successful_attacks"],
        summary["num_failed_attacks"],
        summary["num_skipped"],
    ]
    labels_pie = ["Successful", "Failed", "Skipped"]
    pie_colors = ["#E74C3C", "#2ECC71", "#95A5A6"]
    
    # Remove zero-value slices
    filtered = [(s, l, c) for s, l, c in zip(sizes, labels_pie, pie_colors) if s > 0]
    if filtered:
        sizes_f, labels_f, colors_f = zip(*filtered)
    else:
        sizes_f, labels_f, colors_f = sizes, labels_pie, pie_colors
    
    axes[idx].pie(sizes_f, labels=labels_f, colors=colors_f, autopct="%1.1f%%", startangle=90)
    axes[idx].set_title(f"{name} Model Attack Results")

plt.tight_layout()
plt.savefig("../results/attack_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to results/attack_distribution.png")

## 5. Success Criteria Check

In [ ]:
asr_drop = baseline_asr - augmented_asr
clean_acc_ok = augmented_clean_acc >= 0.89
asr_drop_ok = asr_drop >= 0.15

print("=== Success Criteria ===")
print(f"ASR drop: {asr_drop:.4f} (need >= 0.15)  {'PASS' if asr_drop_ok else 'FAIL'}")
print(f"Augmented clean accuracy: {augmented_clean_acc:.4f} (need >= 0.89)  {'PASS' if clean_acc_ok else 'FAIL'}")